# Logdet Method Profiling Across Matrix Sizes

This notebook profiles the runtime of the log-determinant strategies used in neighbayes.

The profiling matrices come from a regular polygon grid generated by the `neighbayes.dgp` module. Each size `n` is a grid side length producing an `n × n` rook-contiguity layout with `n²` spatial units; the spatial graph is built from the polygon GeoDataFrame returned by the DGP function.

A model names its log-determinant with two arguments: `logdet_method`, the approximation over ρ, and `logdet_decomp`, what supplies its exact values (`eig`, `chol` or `lu`). Configurations compared:

- **`exact` on `eig`**: precompute the eigenvalues of `W` once; evaluate `sum(log(1 - rho * lam_i))` per call.
- **`exact` on `chol`**: one sparse Cholesky (scikit-sparse CHOLMOD) of the D-symmetrised `I - ρW` per call, with the symbolic analysis reused.
- **`cheb` on `eig` / `chol`**: Chebyshev interpolation through exact values at Chebyshev nodes, from the eigenvalues or from sparse Cholesky; evaluation by the Clenshaw recurrence.
- **`aaa` on `chol` / `lu`**: AAA rational interpolation (Nakatsukasa, Sète & Trefethen 2018) through exact values at adaptively selected support points. Rational approximation converges exponentially near singularities, needing ~6–15 support points. `chol` serves undirected `W` and is the auto choice for 500 < n ≤ 60000; `lu` serves directed `W` (KNN, travel time, migration).
- **`stochastic`**: the stochastic Chebyshev expansion of Han, Malioutov & Shin (2015), with no factorization; the auto choice for n > 60000.

For each matrix size, we report:

- **setup time**: build + compile callable logdet function
- **evaluation time**: average cost to evaluate at many rho values

In [ ]:
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytensor
import pytensor.tensor as pt
import scipy.sparse as sp
from libpysal import graph

from neighbayes import dgp
from neighbayes._logdet import logdet_key, make_logdet_fn

In [ ]:
def make_grid_w(n_side: int) -> np.ndarray:
    """Create a row-standardized rook-contiguity matrix from an n_side x n_side polygon grid.

    Uses ``dgp.simulate_sar`` with ``create_gdf=True`` to generate the polygon
    geometry, then builds a contiguity graph from the returned GeoDataFrame.
    """
    gdf = dgp.simulate_sar(n_side=n_side, create_gdf=True)
    W = (
        graph.Graph.build_contiguity(gdf, rook=True)
        .transform("r")
        .sparse.toarray()
        .astype(np.float64)
    )
    return W


def compile_logdet_callable(
    W: np.ndarray,
    method: str,
    rho_min: float,
    rho_max: float,
):
    """Return a compiled callable f(rho) and its setup time in seconds."""
    t0 = time.perf_counter()
    rho = pt.scalar("rho")
    expr = make_logdet_fn(
        W,
        method=method,
        rho_min=rho_min,
        rho_max=rho_max,
    )(rho)
    fn = pytensor.function([rho], expr)
    setup_s = time.perf_counter() - t0
    return fn, setup_s


def bench_eval_seconds(fn, rhos: np.ndarray, repeats: int = 5) -> float:
    """Median per-call evaluation latency in microseconds."""
    run_times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        for r in rhos:
            _ = fn(float(r))
        elapsed = time.perf_counter() - t0
        run_times.append(elapsed / len(rhos))
    return float(np.median(run_times))

In [ ]:
@dataclass
class ProfileConfig:
    # Grid side lengths; obs count = n_side². e.g. 10→100, 20→400, 75→5625.
    sizes: tuple[int, ...] = (10, 20, 25, 40, 50, 75)
    method_specs: tuple[dict, ...] = tuple(
        {
            "label": f"{m} on {d}" if d != "auto" else m,
            "method": m,
            "decomp": d,
            "rho_min": -0.95,
            "rho_max": 0.95,
        }
        for m, d in [
            ("eigenvalue", "auto"),
            ("cheb", "chol"),
            ("aaa", "chol"),
            ("aaa", "lu"),
            ("stochastic", "auto"),
        ]
    )
    method_max_n: dict = None
    eval_points: int = 80
    eval_repeats: int = 3
    seed: int = 2026

    def __post_init__(self):
        if self.method_max_n is None:
            self.method_max_n = {spec["label"]: 75 for spec in self.method_specs}


cfg = ProfileConfig()

results = []
skipped = []

for n in cfg.sizes:
    W = make_grid_w(n_side=n)
    print(f"Profiling n_side={n} ({n * n} obs)...")
    for spec in cfg.method_specs:
        label = spec["label"]
        # The private factories take the internal key the public pair resolves to.
        method = logdet_key(spec["method"], spec["decomp"], W=sp.csr_matrix(W))

        if n > cfg.method_max_n[label]:
            skipped.append(
                {
                    "n_side": n,
                    "n_obs": n * n,
                    "method": label,
                    "reason": "above method_max_n cap",
                }
            )
            continue

        rho_min = spec["rho_min"]
        rho_max = spec["rho_max"]
        rho_grid = np.linspace(rho_min, rho_max, cfg.eval_points)

        try:
            fn, setup_s = compile_logdet_callable(
                W,
                method=method,
                rho_min=rho_min,
                rho_max=rho_max,
            )
            eval_s = bench_eval_seconds(fn, rho_grid, repeats=cfg.eval_repeats)
            results.append(
                {
                    "n_side": n,
                    "n_obs": n * n,
                    "method": label,
                    "logdet_method": method,
                    "rho_min": rho_min,
                    "rho_max": rho_max,
                    "setup_ms": 1e3 * setup_s,
                    "eval_us": 1e6 * eval_s,
                }
            )
        except Exception as exc:
            skipped.append(
                {
                    "n_side": n,
                    "n_obs": n * n,
                    "method": label,
                    "reason": f"failed: {type(exc).__name__}: {exc}",
                }
            )

res = pd.DataFrame(results).sort_values(["method", "n_obs"]).reset_index(drop=True)
if not res.empty:
    res["total_ms"] = res["setup_ms"] + (res["eval_us"] * cfg.eval_points / 1e3)

res

In [ ]:
if res.empty:
    raise RuntimeError("No profiling results were generated.")

# Distinct color + marker per profiled configuration.
method_styles = {
    "eigenvalue": {"color": "#ff7f0e", "marker": "s", "linestyle": "-"},
    "cheb on chol": {"color": "#d62728", "marker": "^", "linestyle": "-"},
    "aaa on chol": {"color": "#2ca02c", "marker": "o", "linestyle": "-"},
    "aaa on lu": {"color": "#9467bd", "marker": "v", "linestyle": "-"},
    "stochastic": {"color": "#1f77b4", "marker": "D", "linestyle": "-"},
}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

for method, grp in res.groupby("method"):
    grp = grp.sort_values("n_obs")
    style = method_styles.get(method, {"marker": "o"})
    axes[0].plot(grp["n_obs"], grp["setup_ms"], label=method, **style)
    axes[1].plot(grp["n_obs"], grp["eval_us"], label=method, **style)
    axes[2].plot(grp["n_obs"], grp["total_ms"], label=method, **style)

axes[0].set_title("Setup Time vs n")
axes[0].set_xlabel("observations")
axes[0].set_ylabel("setup time (ms)")
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Evaluation Time vs n")
axes[1].set_xlabel("observations")
axes[1].set_ylabel("time per rho eval (us)")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)

axes[2].set_title(f"Total Time vs n (setup + {cfg.eval_points} evals)")
axes[2].set_xlabel("observations")
axes[2].set_ylabel("total time (ms)")
axes[2].set_yscale("log")
axes[2].grid(True, alpha=0.3)

# Single shared legend to the right of the figure so it doesn't crowd the panels.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
    title="configuration",
)
plt.show()

In [ ]:
summary = res.pivot_table(
    index="n_obs", columns="method", values=["setup_ms", "eval_us", "total_ms"]
).sort_index()
display(summary)

if skipped:
    skipped_df = (
        pd.DataFrame(skipped).sort_values(["n_side", "method"]).reset_index(drop=True)
    )
    print("Skipped combinations (due to safety caps or failures):")
    display(skipped_df)

## Logdet Approximation Accuracy

This section compares how accurately each interpolating method approximates the true log-determinant curve, independent of sampling noise. We compute the exact log|I - ρW| via eigenvalues and compare against each method's approximation across a dense ρ grid.


In [ ]:
from neighbayes._logdet._aaa import (
    aaa_logdet_eval,
    aaa_logdet_precompute,
)
from neighbayes._logdet._chol_cheb import (
    chol_cheb_logdet_eval_vec,
    chol_cheb_logdet_precompute,
)

# Use the same W from the last profiling run (or regenerate)
W_accuracy = make_grid_w(n_side=25)
W_sp = sp.csr_matrix(W_accuracy)

# Exact logdet via eigenvalues
W_dense = W_sp.toarray()
eigs = np.linalg.eigvals(W_dense)
rho_grid_acc = np.linspace(-0.95, 0.95, 200)
exact_logdet = np.array([np.sum(np.log(np.abs(1 - r * eigs))) for r in rho_grid_acc])

# cheb on chol: exact values by sparse Cholesky at Chebyshev nodes
# Use the empirical range [0.1, 0.8] where the method is most accurate
rho_grid_emp = np.linspace(0.1, 0.8, 200)
exact_logdet_emp = np.array(
    [np.sum(np.log(np.abs(1 - r * eigs))) for r in rho_grid_emp]
)
chol_pre = chol_cheb_logdet_precompute(W_sp, order=None, rho_min=0.1, rho_max=0.8)
chol_approx = chol_cheb_logdet_eval_vec(chol_pre, rho_grid_emp)
chol_err = np.abs(chol_approx - exact_logdet_emp)

# aaa on lu: AAA rational interpolation through sparse-LU values
aaa_pre = aaa_logdet_precompute(W_sp, rho_min=0.1, rho_max=0.8)
aaa_approx = np.array([aaa_logdet_eval(aaa_pre, r) for r in rho_grid_emp])
aaa_err = np.abs(aaa_approx - exact_logdet_emp)

acc_df = pd.DataFrame(
    [
        {
            "method": "cheb on chol",
            "max_err": chol_err.max(),
            "mean_err": chol_err.mean(),
            "rmse": np.sqrt((chol_err**2).mean()),
        },
        {
            "method": "aaa on lu",
            "max_err": aaa_err.max(),
            "mean_err": aaa_err.mean(),
            "rmse": np.sqrt((aaa_err**2).mean()),
        },
    ]
)
display(acc_df)

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

# Empirical range [0.1, 0.8]
ax.plot(
    rho_grid_emp,
    chol_approx - exact_logdet_emp,
    label="cheb on chol",
    alpha=0.7,
    color="#d62728",
)
ax.plot(
    rho_grid_emp,
    aaa_approx - exact_logdet_emp,
    label="aaa on lu",
    alpha=0.7,
    color="#9467bd",
)
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("rho")
ax.set_ylabel("approximation error")
ax.set_title("Logdet Error vs Exact (empirical range [0.1, 0.8])")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

## Coefficient and Fit-Time Comparison Across Logdet Methods

This section uses a regular polygon grid generated by `neighbayes.dgp` to simulate one SAR dataset and estimates the same SAR model with each `logdet_method` and `logdet_decomp`.

We compare:

- posterior mean coefficients (`rho`, `beta_0`, `beta_1`, `beta_2`)
- total wall-clock time to estimate each model

To keep this section runnable in docs contexts, sampling is intentionally modest.


In [ ]:
from neighbayes.models import SAR


def simulate_sar_data(n_side: int = 25, seed: int = 2026):
    """Simulate SAR data on an n_side x n_side polygon grid using the DGP module."""
    rng = np.random.default_rng(seed)
    beta_true = np.array([1.0, 0.8, -0.5], dtype=np.float64)
    rho_true = 0.35
    sigma_true = 0.7

    gdf = dgp.simulate_sar(
        n_side=n_side,
        rho=rho_true,
        beta=beta_true,
        sigma=sigma_true,
        rng=rng,
        create_gdf=True,
    )
    # Keep the SAR parameterization consistent with model assumptions.
    W_graph = graph.Graph.build_contiguity(gdf, rook=True).transform("r")
    y = gdf["y"].to_numpy()
    X_cols = [c for c in gdf.columns if c.startswith("X_")]
    X = gdf[X_cols].to_numpy()
    return gdf, y, X, W_graph, rho_true, beta_true


def fit_sar_for_method(
    y,
    X,
    W,
    method: str,
    decomp: str,
    label: str | None = None,
    draws: int = 1000,
    tune: int = 1000,
    seed: int = 2026,
):
    """Fit SAR with a specific logdet configuration and return posterior means + runtime."""
    t0 = time.perf_counter()

    model = SAR(
        y=y,
        X=X,
        W=W,
        logdet_method=method,
        logdet_decomp=decomp,
    )
    idata = model.fit(
        draws=draws,
        tune=tune,
        chains=2,
        random_seed=seed,
        progressbar=False,
    )

    elapsed_s = time.perf_counter() - t0
    beta_mean = idata.posterior["beta"].mean(("chain", "draw")).to_numpy()
    rho_mean = float(idata.posterior["rho"].mean(("chain", "draw")).to_numpy())

    return {
        "method": label or method,
        "beta_mean": beta_mean,
        "rho_mean": rho_mean,
        "fit_seconds": elapsed_s,
    }


# Build the dataset once and sweep the methods we actually compare.
gdf_model, y_model, X_model, W_model, rho_true, beta_true = simulate_sar_data(n_side=25)

methods_for_model = [
    {"label": "eigenvalue", "method": "eigenvalue", "decomp": "auto"},
    {"label": "cheb on chol", "method": "cheb", "decomp": "chol"},
    {"label": "aaa on chol", "method": "aaa", "decomp": "chol"},
    {"label": "aaa on lu", "method": "aaa", "decomp": "lu"},
    {"label": "stochastic", "method": "stochastic", "decomp": "auto"},
]

fit_rows = []
for spec in methods_for_model:
    print(f"Fitting SAR with {spec['label']} ...")
    fit_rows.append(
        fit_sar_for_method(
            y_model,
            X_model,
            W_model,
            method=spec["method"],
            decomp=spec["decomp"],
            label=spec["label"],
        )
    )


coef_df = pd.DataFrame(fit_rows)

display(coef_df)

## Notes

- `exact` on `eig` carries a one-time O(n³) eigendecomposition cost, then evaluates in O(n) per `rho`: the reference answer at moderate n.
- `exact` on `chol` pays one sparse Cholesky factorization per `rho`, with the symbolic analysis reused: exact at any n where the factorization is affordable, but the most expensive per evaluation.
- `cheb` builds Chebyshev coefficients once from exact values at its nodes (from the eigenvalues, or from sparse Cholesky or LU), then evaluates in O(m) per call.
- `aaa` evaluates exact values at adaptively selected support points, by sparse Cholesky for undirected `W` or sparse LU otherwise, then fits a rational function in barycentric form. It needs only ~6–15 support points, converging exponentially faster than polynomials near singularities.
- `stochastic` needs no factorization; its error is the probes' Monte Carlo error rather than an interpolation error.
- **Parameter ranges:** the timing sweep uses `[-0.95, 0.95]` throughout. The accuracy comparison runs `cheb` on `chol` and `aaa` on `lu` on the empirical range `[0.1, 0.8]`, where they are most accurate (closer to the singularity at ρ=1 needs higher order or more support points).

- Adjust `ProfileConfig.sizes` and `method_max_n` for deeper stress tests.

In [ ]:
import arviz as az
import pandas as pd

# Refit each logdet configuration and compare effective sample sizes for rho.
ess_rows = []
for spec in methods_for_model:
    label = spec["label"]
    model = SAR(
        y=y_model,
        X=X_model,
        W=W_model,
        logdet_method=spec["method"],
        logdet_decomp=spec["decomp"],
        priors={"rho_lower": -1, "rho_upper": 1},
    )
    idata = model.fit(
        draws=1000,
        tune=1000,
        chains=2,
        random_seed=2026,
        progressbar=False,
    )
    summary = az.summary(idata, var_names=["rho"])
    ess = summary.loc["rho", "ess_bulk"]
    ess_rows.append(
        {
            "method": label,
            "ess_rho": ess,
        }
    )
    az.plot_trace_dist(idata, var_names=["rho"], compact=True)
    plt.suptitle(label)

ess_df = pd.DataFrame(ess_rows)
display(ess_df)

ess_df.set_index("method")["ess_rho"].plot.bar(
    ylabel="ESS (rho)", title="Effective Sample Size for rho by Logdet Configuration"
)

## Method selection policy

With `logdet_method=None` and `logdet_decomp="auto"` the selector chooses:

- `eigenvalue` for `n ≤ 500` (one eigendecomposition, then a sum per ρ)
- `aaa` on `chol` for an undirected `W` and `aaa` on `lu` for a directed one, for `500 < n ≤ 60000`, unless the factorization's fill-in would dominate
- `stochastic` for `n > 60000` (no factorization)

Manual overrides:

1. **Default for most work.** Leave both arguments at their defaults.
2. **Eigenvalues at moderate n.** Use `logdet_method="eigenvalue"` when an O(n³) eigendecomposition is affordable.
3. **Chebyshev interpolation.** Use `logdet_method="cheb"`, with the decomposition chosen automatically or named.
4. **Directed W.** `logdet_decomp="auto"` already takes `lu`; name it to be explicit.
5. **Reporting and publication.** Record `logdet_method`, `logdet_decomp` and the `rho` bounds in every benchmark or model report.